In [ ]:
# Install the project dependencies required by this notebook.
%pip install -q pandas numpy duckdb pyarrow scikit-learn xgboost mlflow matplotlib scipy joblib pyyaml

In [ ]:
# Mount Google Drive so Colab can access the private MIMIC-IV files and derived artifacts.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or update the GitHub repository so the notebook can import the shared project code.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mbakos95/aki-sentinel.git"
REPO_DIR = Path("/content/aki-sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Define the private MIMIC-IV and artifact locations used by the pipeline.
from pathlib import Path

MIMIC_ROOT = Path("/content/drive/MyDrive/MIMIC-IV")
HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

PRIVATE_ROOT = Path("/content/drive/MyDrive/AKI-Sentinel-Private")
ARTIFACT_DIR = PRIVATE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the saved models and the project evaluation utilities.
import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd

from src.evaluation import (
    calibration_table,
    choose_threshold_for_sensitivity,
    classification_metrics,
    save_json,
    subgroup_metrics,
)
from src.modeling import get_model_columns

In [ ]:
# Load the validation and test cohorts plus both trained model pipelines.
SPLIT_DIR = ARTIFACT_DIR / "splits"
MODEL_DIR = ARTIFACT_DIR / "models"

val_df = pd.read_parquet(SPLIT_DIR / "validation.parquet")
test_df = pd.read_parquet(SPLIT_DIR / "test.parquet")

logistic = joblib.load(MODEL_DIR / "logistic_regression.joblib")
xgboost = joblib.load(MODEL_DIR / "xgboost.joblib")

In [ ]:
# Generate validation probabilities and choose each model threshold to target at least 85 percent sensitivity.
feature_columns, _, _ = get_model_columns(val_df)

val_lr_prob = logistic.predict_proba(val_df[feature_columns])[:, 1]
val_xgb_prob = xgboost.predict_proba(val_df[feature_columns])[:, 1]

lr_threshold = choose_threshold_for_sensitivity(
    val_df["target"],
    val_lr_prob,
    target_sensitivity=0.85,
)
xgb_threshold = choose_threshold_for_sensitivity(
    val_df["target"],
    val_xgb_prob,
    target_sensitivity=0.85,
)

print("Logistic threshold:", round(lr_threshold, 4))
print("XGBoost threshold:", round(xgb_threshold, 4))

In [ ]:
# Evaluate both frozen models on the later test cohort using thresholds selected only on validation data.
test_lr_prob = logistic.predict_proba(test_df[feature_columns])[:, 1]
test_xgb_prob = xgboost.predict_proba(test_df[feature_columns])[:, 1]

lr_metrics = classification_metrics(
    test_df["target"],
    test_lr_prob,
    lr_threshold,
)
xgb_metrics = classification_metrics(
    test_df["target"],
    test_xgb_prob,
    xgb_threshold,
)

pd.DataFrame(
    [lr_metrics, xgb_metrics],
    index=["Logistic Regression", "XGBoost"],
)

In [ ]:
# Plot the XGBoost calibration curve to compare predicted AKI risk with observed event rates.
calibration = calibration_table(
    test_df["target"],
    test_xgb_prob,
    n_bins=10,
)

plt.figure(figsize=(6, 6))
plt.plot(
    calibration["mean_predicted_probability"],
    calibration["observed_event_rate"],
    marker="o",
    label="XGBoost",
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed AKI rate")
plt.title("AKI Sentinel calibration")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# Evaluate XGBoost performance across sex, age, ICU type, and race subgroups without using race as a model feature.
evaluation_frame = test_df.copy()
evaluation_frame["xgb_probability"] = test_xgb_prob
evaluation_frame["age_group"] = pd.cut(
    evaluation_frame["age"],
    bins=[17, 64, 79, float("inf")],
    labels=["18-64", "65-79", "80+"],
)

sex_metrics = subgroup_metrics(
    evaluation_frame,
    "xgb_probability",
    xgb_threshold,
    "gender",
)
age_metrics = subgroup_metrics(
    evaluation_frame,
    "xgb_probability",
    xgb_threshold,
    "age_group",
)
icu_metrics = subgroup_metrics(
    evaluation_frame,
    "xgb_probability",
    xgb_threshold,
    "first_careunit",
)

display(sex_metrics)
display(age_metrics)
display(icu_metrics)

In [ ]:
# Save aggregate metrics, calibration, thresholds, and private test predictions for monitoring.
EVAL_DIR = ARTIFACT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

save_json(lr_metrics, EVAL_DIR / "test_metrics_logistic_regression.json")
save_json(xgb_metrics, EVAL_DIR / "test_metrics_xgboost.json")
save_json(
    {
        "logistic_regression": lr_threshold,
        "xgboost": xgb_threshold,
    },
    EVAL_DIR / "thresholds.json",
)

calibration.to_csv(EVAL_DIR / "xgboost_calibration.csv", index=False)
sex_metrics.to_csv(EVAL_DIR / "subgroup_gender.csv", index=False)
age_metrics.to_csv(EVAL_DIR / "subgroup_age.csv", index=False)
icu_metrics.to_csv(EVAL_DIR / "subgroup_icu.csv", index=False)

private_predictions = test_df[
    ["subject_id", "hadm_id", "stay_id", "target", "anchor_year_group"]
].copy()
private_predictions["xgb_probability"] = test_xgb_prob
private_predictions.to_parquet(
    EVAL_DIR / "private_test_predictions.parquet",
    index=False,
)